In [86]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_huggingface import HuggingFacePipeline,ChatHuggingFace
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import PydanticOutputParser
import operator
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os
import json
import re


In [87]:
load_dotenv()
hf_token = os.getenv("HUGGINGFACE_API_KEY")

In [88]:
# Common model configuration
pipeline_config = {
    "max_new_tokens": 150,
    "do_sample": False,
    "return_full_text": False
}

# 1. Generator LLM - Qwen
generator_pipeline = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-0.5B-Instruct",
    task="text-generation",
    pipeline_kwargs=pipeline_config
)

generator_llm = ChatHuggingFace(llm=generator_pipeline)


# 2. Evaluator LLM - TinyLlama
evaluator_pipeline = HuggingFacePipeline.from_model_id(
    model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    pipeline_kwargs=pipeline_config
)

evaluator_llm = ChatHuggingFace(llm=evaluator_pipeline)


# 3. Optimizer LLM - SmolLM2
optimizer_pipeline = HuggingFacePipeline.from_model_id(
    model_id="HuggingFaceTB/SmolLM2-1.7B-Instruct",
    task="text-generation",
    pipeline_kwargs=pipeline_config
)

optimizer_llm = ChatHuggingFace(llm=optimizer_pipeline)

Loading weights: 100%|██████████| 218/218 [00:06<00:00, 31.67it/s] 


In [89]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(
        description="Final evaluation result"
    )
    feedback: str = Field(
        description="Feedback for the tweet"
    )

structured_evaluator_llm = PydanticOutputParser(
    pydantic_object=TweetEvaluation
)

In [90]:
#state

class TweetState(TypedDict):

    topic:str
    tweet:str
    evaluation:Literal["approved","needs_improvement"]
    feedback:str
    iteration:int
    max_iteration:int

    tweet_history: Annotated[list[str],operator.add]
    feedback_history: Annotated[list[str],operator.add]

In [91]:
def generate_tweet(state:TweetState):

    #prompt
    messages = [SystemMessage(content="you are a funny and clever Twitter/X influencer."),
                HumanMessage(content= f"""Write a short, original and hilarious tweet on the topic: "{state['topic']}".
                Rules:
                - DO NOT use question-answer format.
                - Max 280 characters.
                - Use observational humor, irony, sarcasm, or cultural references.
                - Think in meme logic, punchines, or relatable takes.
                - Use simple, day to day english.""")]
    # send generator_llm
    response = generator_llm.invoke(messages).content

    # return response
    return {'tweet':response, 'tweet_history':[response]}

In [92]:
def evaluate_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You are a strict Twitter critic."),
        HumanMessage(content=f"""Evaluate this tweet for originality, humor, punchiness, virality and format.
Tweet: "{state['tweet']}"

Reject it if it uses question-answer or setup-punchline format, is over 280 characters, or ends with a weak, generic line.

First line: "evaluation: approved" or "evaluation: needs_improvement".
Then write one paragraph of feedback.""")
    ]
    text = evaluator_llm.invoke(messages).content

    lines = text.strip().lower().splitlines()
    first = lines[0] if lines else ""
    approved = first.strip(' ."') in ("approved", "evaluation: approved")
    if len(state['tweet']) > 280:
        approved = False

    return {'evaluation': "approved" if approved else "needs_improvement",
            'feedback': text,
            'feedback_history': [text]}

In [93]:
def optimize_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You improve tweets based on feedback."),
        HumanMessage(content=f"""Topic: {state['topic']}
Tweet: {state['tweet']}
Feedback: {state['feedback']}

Rewrite the tweet in under 280 characters. Return only the tweet.""")
    ]
    tweet = optimizer_llm.invoke(messages).content.strip()

    return {'tweet': tweet,
            'tweet_history': [tweet],
            'iteration': state['iteration'] + 1}

def route_evaluation(state: TweetState):
    if state['evaluation'] == "approved" or state['iteration'] >= state['max_iteration']:
        return "end"
    return "optimize"

In [94]:
graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', route_evaluation,
                            {"end": END, "optimize": 'optimize'})
graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()

In [95]:
initial_state = {
    "topic": "Indian Railways",
    "iteration": 1,
    "max_iteration": 2
}

# print each step as soon as it finishes
for step in workflow.stream(initial_state, stream_mode="updates"):
    print(step, flush=True)

{'generate': {'tweet': "Why do Indian trains always have to be so slow? Because they're not allowed to go fast enough! 🚧高铁真的太慢了，它们只能慢悠悠地行驶。高铁的速度比地铁快多了！高铁真是个“慢火车”。 😂", 'tweet_history': ["Why do Indian trains always have to be so slow? Because they're not allowed to go fast enough! 🚧高铁真的太慢了，它们只能慢悠悠地行驶。高铁的速度比地铁快多了！高铁真是个“慢火车”。 😂"]}}


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'evaluate': {'evaluation': 'needs_improvement', 'feedback': 'Originality: This tweet is original in its content, as it criticizes the slowness of Indian trains. The use of the word "slow" in the tweet is unique and adds to its originality.\n\nHumor: The tweet is humorous in its punchline, which is a play on the phrase "high-speed rail" and the fact that Indian trains are not allowed to go fast enough. The tweet also includes a pun on the word "slow" and the fact that Indian trains are slow.\n\nPunchiness: The tweet is punchy in its delivery, as it is short and to the point. The tweet is also punchy in its use of ex', 'feedback_history': ['Originality: This tweet is original in its content, as it criticizes the slowness of Indian trains. The use of the word "slow" in the tweet is unique and adds to its originality.\n\nHumor: The tweet is humorous in its punchline, which is a play on the phrase "high-speed rail" and the fact that Indian trains are not allowed to go fast enough. The twee

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'evaluate': {'evaluation': 'needs_improvement', 'feedback': 'Originality: This tweet is original in its content, as it criticizes the slowness of Indian trains. The use of the word "slow" in the tweet is unique and adds to its originality.\n\nHumor: The tweet is humorous in its punchline, which is a play on the phrase "high-speed rail" and the fact that Indian trains are not allowed to go fast enough. The tweet also includes a pun on the word "slow" and the fact that Indian trains are slow.\n\nPunchiness: The tweet is punchy in its delivery, as it is short and to the point. The tweet is also punchy in its use of ex', 'feedback_history': ['Originality: This tweet is original in its content, as it criticizes the slowness of Indian trains. The use of the word "slow" in the tweet is unique and adds to its originality.\n\nHumor: The tweet is humorous in its punchline, which is a play on the phrase "high-speed rail" and the fact that Indian trains are not allowed to go fast enough. The twee